# Team run guide — Problem B agent

Run these cells **top to bottom**. Each step shows the command you would type,
then runs it so you see the output inline. No installs, no API key, no network —
everything here is the free scripted backend.

For *what the system does and why*, read `docs/HOW_IT_WORKS.pdf` first. This
notebook is for operating it.

> **Kernel working directory must be `A2_scaffold/`** (where this file lives).
> If a cell can't find a module or the data, that's why — restart the kernel
> with this folder as the working directory.

## 0 · Setup — confirm it can find everything

No command for this one. It just imports the config and prints which backend,
problem and decision mode are active, and where it found the reference data.
If the data path fails, the error tells you exactly how to fix it.

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())          # make the modules importable
import config
print(config.summary())
print("data:", config.data_root())

## 1 · Confirm it runs — the marker's path

```bash
python3 run_eval.py
```

Runs every case that has a script, grades it against the answer key, writes
`results.json`. This is exactly what a marker types after cloning. It must work
with no arguments.

In [ ]:
import run_eval
run_eval.main(["run_eval.py"])

## 2 · One case, every turn shown

```bash
python3 run_eval.py REF-5602
```

Pass a case id to watch the loop step by step — each turn, each tool call, the
decision record, and whether it passed the code check.

In [ ]:
run_eval.main(["run_eval.py", "REF-5602"])

## 3 · The decision-mode switch

```bash
python3 run_eval.py --mode rules REF-5602
python3 run_eval.py --mode model REF-5602
```

`--mode` overrides `config.DECISION_MODE` for that run only — it never edits
`config.py`. On this case both modes agree (a clean booking). What differs is
what the code computes and checks underneath — see step 5.

- **rules** — the four gates are resolved by `tools.resolve_routing()`; the
  model just reports the result.
- **model** — the model reads the same raw facts and applies the gates itself.

In [ ]:
run_eval.main(["run_eval.py", "--mode", "rules", "REF-5602"])

In [ ]:
run_eval.main(["run_eval.py", "--mode", "model", "REF-5602"])

### Prefer a guided pick?

```bash
python3 choose_mode.py
```

Explains both modes in plain language and asks which to run. It's a wrapper
around `run_eval.py` — same code path, just asks first. (Not run here because
it waits for keyboard input.)

## 4 · See exactly what the model is told

```bash
python3 run_eval.py --mode rules --prompt
python3 run_eval.py --mode model --prompt
```

Prints the full system prompt for that mode and its token size, then stops.
The two modes send different routing-rule text — this is the v1/v2 surface for
D2(b). The scripted backend never reads this; it only matters live.

In [ ]:
run_eval.main(["run_eval.py", "--mode", "rules", "--prompt"])

In [ ]:
run_eval.main(["run_eval.py", "--mode", "model", "--prompt"])

## 5 · The safety net firing — REF-6007

A referral where every real fact is clean, so `resolve_routing()` resolves it
to **book** — but the free text claims a consultant already approved it. The
agent must escalate on that basis, and the route-consistency guardrail logs the
disagreement between the code's answer and the agent's.

This runs the case directly so we can print the guardrail log.

In [ ]:
import config, harness
config.DECISION_MODE = "rules"        # agent.py reads this at run time
results, _ = harness.run_set(["REF-6007"], verbose=True)
rec = results[0]["record"]
print()
print("decision        :", rec["decision"])
print("resolved_routing:", rec["resolved_routing"], "  <- what the four gates alone say")
print("guardrails_fired:", rec["guardrails_fired"])
print("code check      :", "PASS" if results[0]["passed"] else "FAIL", results[0]["fails"])

## 6 · Check the data hangs together

```bash
python3 ../A2_reference_data/check_my_data.py
```

Run this after **every** change to a fixture or a label. It catches: an id that
resolves to nothing, a shipped row that was edited, a duplicate id, and a case
with no label (or a label with no case). It does **not** check whether a label
is *correct* — that judgement is yours.

In [ ]:
import subprocess
print(subprocess.run(
    ["python3", "../A2_reference_data/check_my_data.py"],
    capture_output=True, text=True).stdout)

## 7 · The D7 loop-failure demo

```bash
python3 demo_loop_failure.py
```

The shipped worked example of D7's method: take the working agent, delete one
guard (action de-duplication), show the run burns turns and cost **without
raising any exception** and still returns the right answer — visible only
because turns and cost are instrumented. Then it puts the guard back.

In [ ]:
import demo_loop_failure
demo_loop_failure.main()

## 8 · Add your own evaluation case

No cell for this — it edits files. The loop, in `A2_reference_data/`:

```bash
# 1 · edit the EXTRA_* lists near the bottom of make_fixtures_B.py
python3 make_fixtures_B.py          # 2 · regenerate data_B/
python3 check_my_data.py            # 3 · fails: your new case has no label
# 4 · add the label to expected_outcomes_B.json BY HAND, from Appendix A's
#     routing table - BEFORE you run the agent on it
python3 check_my_data.py            # 5 · "Your data hangs together."
```

Then, in `A2_scaffold/`:

```bash
# 6 · add a SCRIPTS entry in backends.py so it runs on the free backend
python3 run_eval.py --mode rules  <your_case_id>
python3 run_eval.py --mode model  <your_case_id>
```

Two fully worked examples (a duplicate-history case and a boundary case) are in
`docs/DATA_NOTES.md §2`, and the labelling rules are in that file's §4. The
single most important rule: **write the label from the routing table before you
run the agent**, never from what the agent produced.

## The file map

| File | What it is | You'll touch it? |
|---|---|---|
| `config.py` | `BACKEND`, `MODEL`, `DECISION_MODE`, guardrail limits, prices | Set `DECISION_MODE`; leave `BACKEND=scripted` as the committed default |
| `tools.py` | The tool layer + `resolve_routing()` (the four gates as code) + six-field descriptors | Heavily — this is D2 |
| `resolve_routing()` inside `tools.py` | Appendix A's routing table as a pure function. **Not** in the tool registry — `agent.py` calls it directly | Only if the routing table itself is misread |
| `agent.py` | The ReAct loop, instrumented; computes `resolve_routing` every run and runs the consistency check | Read every line before changing |
| `guardrails.py` | Step cap, budget, de-dup, autonomy gate, **route consistency** | The limits are yours to set from evidence |
| `prompt.py` | Assembles descriptors + routing rules into the system prompt; two variants for Problem B (`B_rules` / `B_model`) | Yes — this is D2(b) |
| `backends.py` | Scripted backend (hand-written move sequences per case) + the live one | Add a `SCRIPTS` entry per new case |
| `harness.py` | Load, run, code check, judgement queue, report | Some |
| `run_eval.py` | Entry point. What a marker runs. `--mode` flag added | Rarely |
| `choose_mode.py` | Guided mode picker for the team; wraps `run_eval.py` | No |
| `demo_loop_failure.py` | D7's method, worked once | Copy the method for the second failure |
| `../A2_reference_data/make_fixtures_B.py` | Fixture generator; edit the `EXTRA_*` lists only | Yes — your new cases |
| `../A2_reference_data/expected_outcomes_B.json` | The answer key. Shipped 15 + our additions | Yes — one label per new case, by hand |
| `../A2_reference_data/check_my_data.py` | Integrity checker | Never edit; run constantly |

## Rules you cannot break

- **Never edit or delete a shipped fixture row.** Add new rows with new ids
  (`REF-6001+`, `P-2001+`). `check_my_data.py` fingerprints every shipped row
  and will name the one that moved.
- **Never touch** `URGENCY_BANDS`, the shipped specialties' `mandatory_tests` /
  `red_flag_terms`, or `AS_OF`. Those are the protocol, not ours to rewrite.
  Adding a *new* specialty is fine.
- **Write every label from Appendix A's routing table, before running the
  agent.** A label copied from the agent's output measures nothing.
- **`BACKEND = "scripted"` stays the committed default.** A marker clones and
  runs it that way; if it doesn't reproduce, Technical Execution is capped.
- **Every number in the report is one you ran yourself.** Not an estimate
  called a measurement, not something an AI produced.

## Git workflow

- Branch per piece of work; don't commit straight to `main` once the team is
  all pushing.
- `results.json` and `__pycache__/` are git-ignored — don't force them in.
- **Commit your own cases under your own name.** Section 8 of the brief leans
  on commit history to corroborate `CONTRIBUTIONS.md`.
- Commit message trailer for anything built with AI assistance:
  `Co-Authored-By: Claude Sonnet 5 <noreply@anthropic.com>`, and note the
  assistance in `CONTRIBUTIONS.md` and the self-appraisal's AI-use declaration.
- `TEAM_GUIDE` (this notebook) is *how to work here*. `CONTRIBUTIONS.md` is the
  *record of who did what*. Different files — don't edit the wrong one.

## What's still open

- **Guardrail checklist (D3b)** — separate deliverable, 10+ cases, ≥3 hostile
  text, each naming the wrong behaviour it catches. **Zero written.**
- **Judgement check (D4)** — a person (or a declared second model) reads each
  `reason` field against its `must_record` list. Not started.
- **Live battery (D5b)** — `BACKEND="live"` has never run. Needs a key.
- **Second D7 failure** — the `REF-6007` bug/fix is a strong candidate;
  currently only written up in `docs/DATA_NOTES.md`, not a runnable demo.
- **More eval cases** — coverage vs. targets is in `docs/DATA_NOTES.md §5`.